# SynapseGPT Outlier Probe

Measures activation, gradient, and weight outliers in your **trained** SynapseGPT model.
Loads the latest checkpoint from Google Drive, runs forward+backward on a small synthetic batch,
and reports per-layer outlier statistics.

**Outlier definition**: For any tensor $x$:
- `rms = sqrt(mean(x^2))`
- `max_abs = max(abs(x))`
- `max_over_rms = max_abs / rms` (primary ranking metric)

Also reports: `mean_abs`, `p99_abs`, `p999_abs`, `p99_over_rms`, `p999_over_rms`.

**Gradients** are captured *before* gradient clipping.

All results auto-download as CSV + JSON when the notebook finishes.

In [ ]:
# @title 1. Clone repo, install deps
import os, sys, math, json, glob, gc, csv
from collections import defaultdict
from datetime import datetime
import numpy as np

REPO_URL = "https://github.com/ajencinas/synapse.git"
if not os.path.isdir("synapse"):
    !git clone {REPO_URL}
else:
    print("Already cloned.")

# Add tests/ to path so we can import synapse_model (not train.py!)
sys.path.insert(0, "./synapse/tests")

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"  GPU: {props.name} | VRAM: {props.total_memory / 1024**3:.1f} GiB")

def vram(label=""):
    if device.type == "cuda":
        used = torch.cuda.memory_allocated() / 1024**3
        peak = torch.cuda.max_memory_allocated() / 1024**3
        print(f"  VRAM [{label}]: {used:.2f} GiB used, {peak:.2f} GiB peak")
    else:
        print(f"  VRAM [{label}]: N/A (CPU)")

In [ ]:
# @title 2. Mount Google Drive and find latest checkpoint
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

SYNAPSE_DIR = "/content/drive/MyDrive/synapse"
CHECKPOINT_DIR = os.path.join(SYNAPSE_DIR, "checkpoints")
EXPORT_DIR = "/content/synapse_outlier_results"
os.makedirs(EXPORT_DIR, exist_ok=True)

if not os.path.isdir(CHECKPOINT_DIR):
    raise FileNotFoundError(f"Checkpoint dir not found: {CHECKPOINT_DIR}")

ckpt_files = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "*.pth")),
                    key=os.path.getmtime, reverse=True)

if not ckpt_files:
    raise FileNotFoundError(f"No .pth files found in {CHECKPOINT_DIR}")

print(f"Found {len(ckpt_files)} checkpoint(s). Latest 5:")
for i, p in enumerate(ckpt_files[:5]):
    size_gb = os.path.getsize(p) / 1024**3
    mtime = datetime.fromtimestamp(os.path.getmtime(p)).strftime("%Y-%m-%d %H:%M")
    print(f"  [{i}] {os.path.basename(p)}  ({size_gb:.2f} GB, {mtime})")

CHECKPOINT_PATH = ckpt_files[0]
print(f"\nSelected: {os.path.basename(CHECKPOINT_PATH)}")

In [ ]:
# @title 3. Read vocab size and show checkpoint metadata
META_PATH = os.path.join(SYNAPSE_DIR, "token_shards_merged", "meta.json")
if os.path.exists(META_PATH):
    with open(META_PATH) as f:
        meta = json.load(f)
    VOCAB_SIZE_RAW = int(meta["vocab_size"])
    VOCAB_SIZE = VOCAB_SIZE_RAW
    if VOCAB_SIZE % 64 != 0:
        VOCAB_SIZE = ((VOCAB_SIZE + 63) // 64) * 64
    print(f"Vocab: raw={VOCAB_SIZE_RAW}  padded={VOCAB_SIZE}")
else:
    VOCAB_SIZE = 131072
    print(f"meta.json not found, defaulting vocab_size={VOCAB_SIZE}")

# Inspect checkpoint metadata without loading weights
print("\nCheckpoint metadata:")
ckpt_obj = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
if isinstance(ckpt_obj, dict) and ckpt_obj.get("schema") == "v2":
    print(f"  Format:      v2")
    print(f"  Step:        {ckpt_obj.get('curr_step', '?')}")
    eval_hist = ckpt_obj.get("eval_history", [])
    if eval_hist:
        print(f"  Eval points: {len(eval_hist)}")
        last = eval_hist[-1]
        print(f"  Last eval:   step={last.get('step','?')}, loss={last.get('loss','?'):.4f}")
        by_src = last.get("by_source", {})
        if by_src:
            print(f"  Per-source losses:")
            for src, val in sorted(by_src.items()):
                print(f"    {src}: {val:.4f}")
    last_eval = ckpt_obj.get("last_eval_loss")
    if last_eval is not None:
        print(f"  Final eval:  {last_eval:.4f}")
    seen = ckpt_obj.get("seen_shards", [])
    print(f"  Seen shards: {len(seen)} (unique={len(set(seen))})")
elif isinstance(ckpt_obj, dict) and "model" in ckpt_obj:
    print("  Format: dict with 'model' key (no metadata)")
else:
    print("  Format: legacy (plain state_dict, no metadata)")

del ckpt_obj
gc.collect()

In [ ]:
# @title 4. Build model and load trained weights

from synapse_model import (
    SynapseGPT, TransformerBlock, CausalGQA, SwiGLU, RMSNorm,
    precompute_rope, apply_rope,
    EMBED_DIM, NUM_LAYERS, NUM_HEADS, NUM_KV_HEADS,
    FF_HIDDEN_DIM, RMSNORM_EPS, BLOCK_SIZE, ROPE_BASE,
)

model = SynapseGPT(VOCAB_SIZE)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built: {n_params/1e9:.3f}B params")
print(f"  embed={EMBED_DIM}, layers={NUM_LAYERS}, heads={NUM_HEADS}, kv_heads={NUM_KV_HEADS}")

# Load checkpoint weights
ckpt_obj = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
if isinstance(ckpt_obj, dict) and ckpt_obj.get("schema") == "v2":
    state_dict = ckpt_obj["model"]
elif isinstance(ckpt_obj, dict) and "model" in ckpt_obj:
    state_dict = ckpt_obj["model"]
else:
    state_dict = ckpt_obj

new_state = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model_state = model.state_dict()
safe_state = {
    k: v for k, v in new_state.items()
    if k in model_state and v.shape == model_state[k].shape
}
model.load_state_dict(safe_state, strict=False)
print(f"Loaded {len(safe_state)}/{len(model_state)} layers")

del ckpt_obj, state_dict, new_state, model_state, safe_state
gc.collect()

model = model.to(device)
model.train()
vram("after model load")

print("\nModel architecture:")
for i, block in enumerate(model.blocks):
    attn = block.attn
    ff = block.ff
    print(f"  Block {i:>2}: Attn(q={attn.q_proj.weight.shape}, "
          f"k={attn.k_proj.weight.shape}, v={attn.v_proj.weight.shape}, "
          f"o={attn.o_proj.weight.shape}) | "
          f"SwiGLU(w1={ff.w1.weight.shape}, w2={ff.w2.weight.shape})")

In [ ]:
# @title 5. Outlier measurement utilities

def tensor_outlier_stats(x: torch.Tensor, name: str = "") -> dict:
    flat = x.detach().float().view(-1)
    if flat.numel() == 0:
        return {"name": name, "numel": 0}
    rms = torch.sqrt(flat.pow(2).mean()).item()
    max_abs = flat.abs().max().item()
    mean_abs = flat.abs().mean().item()
    sorted_abs = flat.abs().sort().values
    n = sorted_abs.numel()
    p99_abs = sorted_abs[int(0.99 * n)].item()
    p999_abs = sorted_abs[min(int(0.999 * n), n - 1)].item()
    max_over_rms = max_abs / rms if rms > 1e-12 else float("inf")
    p99_over_rms = p99_abs / rms if rms > 1e-12 else float("inf")
    p999_over_rms = p999_abs / rms if rms > 1e-12 else float("inf")
    return {
        "name": name, "numel": flat.numel(),
        "rms": round(rms, 6), "max_abs": round(max_abs, 6),
        "mean_abs": round(mean_abs, 6),
        "p99_abs": round(p99_abs, 6), "p999_abs": round(p999_abs, 6),
        "max_over_rms": round(max_over_rms, 4),
        "p99_over_rms": round(p99_over_rms, 4),
        "p999_over_rms": round(p999_over_rms, 4),
    }


def outlier_table(stats_list, top_k=25):
    valid = [s for s in stats_list if s.get("numel", 0) > 0]
    valid.sort(key=lambda s: s["max_over_rms"], reverse=True)
    return valid[:top_k]


def print_outlier_table(stats_list, title="Outlier Table"):
    print(f"\n{'=' * 130}")
    print(f"  {title}")
    print(f"{'=' * 130}")
    hdr = (f"{'#':>3} | {'Name':<55} | {'RMS':>9} | {'Max/RMS':>8} | "
            f"{'MaxAbs':>9} | {'P99/RMS':>8} | {'P999/RMS':>8} | {'MeanAbs':>9} | {'Numel':>8}")
    print(hdr)
    print("-" * 130)
    for i, s in enumerate(stats_list):
        line = (f"{i+1:>3} | {s['name']:<55} | {s['rms']:>9.4f} | {s['max_over_rms']:>8.2f} | "
                f"{s['max_abs']:>9.4f} | {s['p99_over_rms']:>8.2f} | {s['p999_over_rms']:>8.2f} | "
                f"{s['mean_abs']:>9.4f} | {s['numel']:>8}")
        print(line)
    print()

In [ ]:
# @title 6. Register forward hooks on ALL blocks

activation_store = {}

def make_hook(name):
    def hook(module, inp, out):
        if isinstance(out, (tuple, list)):
            out = out[0]
        activation_store[name] = out.detach()
    return hook

hooks = []

# Hook every block's main output
for i, block in enumerate(model.blocks):
    hooks.append(block.register_forward_hook(make_hook(f"block_{i:02d}.output")))

# Hook ALL blocks' internals: norm1, attn, attn.o_proj, norm2, ff
for i, block in enumerate(model.blocks):
    hooks.append(block.norm1.register_forward_hook(make_hook(f"block_{i:02d}.norm1")))
    hooks.append(block.attn.register_forward_hook(make_hook(f"block_{i:02d}.attn")))
    hooks.append(block.attn.o_proj.register_forward_hook(make_hook(f"block_{i:02d}.attn.o_proj")))
    hooks.append(block.norm2.register_forward_hook(make_hook(f"block_{i:02d}.norm2")))
    hooks.append(block.ff.register_forward_hook(make_hook(f"block_{i:02d}.ff")))

# Final norm + lm_head
hooks.append(model.final_norm.register_forward_hook(make_hook("final_norm")))
hooks.append(model.lm_head.register_forward_hook(make_hook("lm_head")))

print(f"Hooked {len(hooks)} points: {len(model.blocks)} blocks x 5 internals + 2 final")

In [ ]:
# @title 7. Run N forward passes and average activation stats
# Running multiple passes gives more stable activation outlier measurements
# because different random inputs may or may not trigger rare neurons.

N_PASSES = 3
B = 1
T = min(512, BLOCK_SIZE)

accumulated_act_stats = defaultdict(list)

print(f"Running {N_PASSES} forward passes (batch={B}, seq={T})...")
for pass_idx in range(N_PASSES):
    torch.manual_seed(42 + pass_idx)
    xb = torch.randint(0, VOCAB_SIZE, (B, T), device=device)
    yb = torch.randint(0, VOCAB_SIZE, (B, T), device=device)

    # Clear previous activations
    activation_store.clear()

    with torch.amp.autocast("cuda" if device.type == "cuda" else "cpu", dtype=torch.bfloat16):
        logits = model(xb)
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yb.view(-1))

    print(f"  Pass {pass_idx+1}: loss={loss.item():.4f}")

    # Collect stats for this pass
    for name, act in activation_store.items():
        accumulated_act_stats[name].append(tensor_outlier_stats(act, name))

    # Free memory
    del logits, loss, xb, yb
    torch.cuda.empty_cache() if device.type == "cuda" else None

# Average across passes: RMS, mean_abs are averaged; max/rms computed from averaged values
activation_stats = []
for name, pass_stats in accumulated_act_stats.items():
    n = len(pass_stats)
    avg = {
        "name": name,
        "numel": pass_stats[0]["numel"],
        "rms": round(sum(s["rms"] for s in pass_stats) / n, 6),
        "max_abs": round(max(s["max_abs"] for s in pass_stats), 6),
        "mean_abs": round(sum(s["mean_abs"] for s in pass_stats) / n, 6),
        "p99_abs": round(sum(s["p99_abs"] for s in pass_stats) / n, 6),
        "p999_abs": round(sum(s["p999_abs"] for s in pass_stats) / n, 6),
    }
    avg["max_over_rms"] = round(avg["max_abs"] / avg["rms"], 4) if avg["rms"] > 1e-12 else float("inf")
    avg["p99_over_rms"] = round(avg["p99_abs"] / avg["rms"], 4) if avg["rms"] > 1e-12 else float("inf")
    avg["p999_over_rms"] = round(avg["p999_abs"] / avg["rms"], 4) if avg["rms"] > 1e-12 else float("inf")
    activation_stats.append(avg)

print(f"\nCollected activation stats from {N_PASSES} passes.")
vram("after activations")

In [ ]:
# @title 8. Run backward pass for gradients (BEFORE any clipping)
# Use the LAST forward pass's loss so gradients flow through trained weights

activation_store.clear()
torch.manual_seed(42)
xb = torch.randint(0, VOCAB_SIZE, (B, T), device=device)
yb = torch.randint(0, VOCAB_SIZE, (B, T), device=device)

with torch.amp.autocast("cuda" if device.type == "cuda" else "cpu", dtype=torch.bfloat16):
    logits = model(xb)
    loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), yb.view(-1))

loss.backward()
print(f"loss={loss.item():.4f} | backward complete (before any grad clipping)")

# Gradient stats
grad_stats = []
grad_count = 0
for name, param in model.named_parameters():
    if param.grad is None:
        continue
    grad_stats.append(tensor_outlier_stats(param.grad, name))
    grad_count += 1

print(f"Collected gradient stats for {grad_count} parameters.")
vram("after backward")

In [ ]:
# @title 9. Weight outlier stats

weight_stats = []
for name, param in model.named_parameters():
    weight_stats.append(tensor_outlier_stats(param.data, name))

print(f"Collected weight stats for {len(weight_stats)} parameters.")
vram("after weight stats")

In [ ]:
# @title 10. Print top outliers for all three categories

top_act = outlier_table(activation_stats, top_k=25)
top_grad = outlier_table(grad_stats, top_k=25)
top_w = outlier_table(weight_stats, top_k=25)

print_outlier_table(top_act, "Top Activation Outliers (by Max/RMS)")
print_outlier_table(top_grad, "Top Gradient Outliers (by Max/RMS)")
print_outlier_table(top_w, "Top Weight Outliers (by Max/RMS)")

In [ ]:
# @title 11. Bar plots of outlier ratios
try:
    import matplotlib.pyplot as plt

    def plot_outlier_bars(stats_list, title, top_k=20, ax=None):
        top = stats_list[:top_k][::-1]
        names = [s["name"][:50] for s in top]
        values = [s["max_over_rms"] for s in top]
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, max(4, len(names) * 0.35)))
        colors = ["crimson" if v > 20 else "darkorange" if v > 10 else "steelblue" for v in values]
        ax.barh(names, values, color=colors)
        ax.set_xlabel("Max / RMS (outlier ratio)")
        ax.set_title(title)
        ax.axvline(x=10, color="darkorange", linestyle="--", alpha=0.5, label="mild (>10)")
        ax.axvline(x=20, color="crimson", linestyle="--", alpha=0.5, label="severe (>20)")
        ax.legend(loc="lower right", fontsize=8)
        ax.grid(axis="x", alpha=0.3)
        return ax

    fig, axes = plt.subplots(1, 3, figsize=(24, 12))
    plot_outlier_bars(top_act, "Activation Outliers", top_k=20, ax=axes[0])
    plot_outlier_bars(top_grad, "Gradient Outliers", top_k=20, ax=axes[1])
    plot_outlier_bars(top_w, "Weight Outliers", top_k=20, ax=axes[2])
    plt.tight_layout()
    plt.savefig(os.path.join(EXPORT_DIR, "outlier_barplots.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {EXPORT_DIR}/outlier_barplots.png")

except ImportError:
    print("matplotlib not available, skipping plots.")

In [ ]:
# @title 12. Per-layer trend plot (Max/RMS across all layers)
try:
    import matplotlib.pyplot as plt

    # Group by block index for all three categories
    def extract_block_trends(stats_list, num_layers):
        block_act = []
        block_grad = []
        block_w = []
        for blk in range(num_layers):
            def avg_for(prefix, category):
                if category == "act":
                    vals = [s["max_over_rms"] for s in activation_stats if s["name"].startswith(f"block_{blk:02d}")]
                elif category == "grad":
                    vals = [s["max_over_rms"] for s in grad_stats if f"blocks.{blk}." in s["name"]]
                else:
                    vals = [s["max_over_rms"] for s in weight_stats if f"blocks.{blk}." in s["name"]]
                return max(vals) if vals else 0
            block_act.append(avg_for(None, "act"))
            block_grad.append(avg_for(None, "grad"))
            block_w.append(avg_for(None, "wt"))
        return block_act, block_grad, block_w

    act_trend, grad_trend, wt_trend = extract_block_trends(activation_stats, NUM_LAYERS)
    layers = list(range(NUM_LAYERS))

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(layers, act_trend, "o-", label="Activations", color="steelblue")
    ax.plot(layers, grad_trend, "s-", label="Gradients", color="crimson")
    ax.plot(layers, wt_trend, "^-", label="Weights", color="green")
    ax.axhline(y=10, color="darkorange", linestyle="--", alpha=0.4)
    ax.axhline(y=20, color="crimson", linestyle="--", alpha=0.4)
    ax.set_xlabel("Block Index")
    ax.set_ylabel("Max / RMS")
    ax.set_title("Outlier Ratio (Max/RMS) Across Layers")
    ax.set_xticks(layers)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(EXPORT_DIR, "outlier_trend.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {EXPORT_DIR}/outlier_trend.png")

except ImportError:
    print("matplotlib not available, skipping plots.")

In [ ]:
# @title 13. Per-layer summary

print("=" * 80)
print("  Per-Block Summary: Max Max/RMS across sub-modules in each block")
print("=" * 80)
for blk in range(NUM_LAYERS):
    def blk_max(prefix, cat):
        if cat == "act":
            vals = [s["max_over_rms"] for s in activation_stats if s["name"].startswith(f"block_{blk:02d}")]
        elif cat == "grad":
            vals = [s["max_over_rms"] for s in grad_stats if f"blocks.{blk}." in s["name"]]
        else:
            vals = [s["max_over_rms"] for s in weight_stats if f"blocks.{blk}." in s["name"]]
        return f"{max(vals):.2f}" if vals else "---"
    print(f"  Block {blk:>2}: act={blk_max(None,'act'):>8}  |  grad={blk_max(None,'grad'):>8}  |  wt={blk_max(None,'wt'):>8}")
print()

In [ ]:
# @title 14. Export all results to CSV and JSON


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = os.path.join(EXPORT_DIR, f"outlier_results_{timestamp}.csv")
json_path = os.path.join(EXPORT_DIR, f"outlier_results_{timestamp}.json")

# CSV: flat table with all three categories
csv_rows = []
for cat, stats in [("activation", activation_stats),
                   ("gradient", grad_stats),
                   ("weight", weight_stats)]:
    for s in stats:
        row = {"category": cat, **s}
        csv_rows.append(row)

csv_rows.sort(key=lambda r: r["max_over_rms"], reverse=True)

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=csv_rows[0].keys())
    writer.writeheader()
    writer.writerows(csv_rows)

# JSON: structured by category
json_data = {
    "checkpoint": os.path.basename(CHECKPOINT_PATH),
    "model_params": n_params,
    "vocab_size": VOCAB_SIZE,
    "block_size": BLOCK_SIZE,
    "num_layers": NUM_LAYERS,
    "embed_dim": EMBED_DIM,
    "n_activation_stats": len(activation_stats),
    "n_gradient_stats": len(grad_stats),
    "n_weight_stats": len(weight_stats),
    "activation": activation_stats,
    "gradient": grad_stats,
    "weight": weight_stats,
    "top_activation": top_act,
    "top_gradient": top_grad,
    "top_weight": top_w,
}

with open(json_path, "w") as f:
    json.dump(json_data, f, indent=2)

print(f"Exported:")
print(f"  CSV:  {csv_path}")
print(f"  JSON: {json_path}")
print(f"  Rows: {len(csv_rows)} total")
print(f"  Size: {os.path.getsize(csv_path)/1024:.1f} KB (csv), {os.path.getsize(json_path)/1024:.1f} KB (json)")

In [ ]:
# @title 15. Auto-download all results from Colab
from google.colab import files

print("Downloading results to your machine...")
for fname in os.listdir(EXPORT_DIR):
    fpath = os.path.join(EXPORT_DIR, fname)
    if os.path.isfile(fpath):
        print(f"  -> {fname}")
        files.download(fpath)

print("\nAll files downloaded.")

In [ ]:
# @title 16. Interpretation

print("""
========================= INTERPRETATION =========================

Metric definitions
------------------
  RMS       = sqrt(mean(x^2))       -- typical scale of the tensor
  Max/RMS   = max_abs / rms          -- how many sigma the largest element is
  P99/RMS   = 99th percentile / rms  -- tail extremity (less noisy than max)
  P999/RMS  = 99.9th percentile / rms

What high values mean
---------------------
Max/RMS ~ 3-5     Normal. Typical for Gaussian-like distributions.
Max/RMS ~ 6-10    Mild outliers. A few activations/weights are noticeably
                  larger than the bulk distribution.
Max/RMS ~ 10-30   Moderate outliers. Common in early-training transformer
                  FFN layers before the model learns to self-stabilize.
Max/RMS > 30      Severe outliers. May indicate numerical instability or
                  a "magnet" neuron that dominates the update. Could lead
                  to loss spikes, especially if gradients are also high.

Gradient outliers deserve special attention:
  - High grad Max/RMS with low weight Max/RMS  ->  intermittent large updates
    that may destabilize training (watch for loss spikes).
  - High grad AND weight Max/RMS  ->  a persistent outlier feature that the
    model relies on heavily. May limit quantizability (QLoRA / SmoothQuant).

Activation outliers:
  - Concentrated in SwiGLU (ff) outputs: expected -- the gating mechanism can
    produce large positive values from SiLU(x) * linear projection.
  - In attention o_proj: may indicate that a few token positions dominate
    the attention-weighted sum (softmax not diffuse enough).
  - In RMSNorm outputs: RMSNorm explicitly rescales to RMS ~ 1, so
    Max/RMS ~ max_abs (since rms ~ 1). Values > 10 here are noteworthy.

Implications for quantization (e.g. INT8 / FP8 training):
  - Layers with Max/RMS > 10 need per-tensor or per-channel scaling factors
    to avoid clipping the tail.
  - Layers with Max/RMS > 30 may require mixed-precision (keeping those
    activations in FP16/BF16) even in an otherwise quantized pipeline.
==================================================================
""")

In [ ]:
# @title 17. Cleanup
for h in hooks:
    h.remove()
del activation_store, accumulated_act_stats
gc.collect()
torch.cuda.empty_cache() if device.type == "cuda" else None
vram("final")
print(f"\nDone. {len(hooks)} hooks removed.")